# 👄 Tu app de lectura de labios — versión UN BOTÓN

**Solo tienes que hacer UNA cosa:** toca el **▶** de la celda de abajo y **espera 10–15 minutos**
sin cerrar la página. Cuando termine, abajo del todo saldrá un enlace azul tipo
`https://xxxx.gradio.live` → tócalo y **esa es tu app** (funciona en el navegador del móvil).

- No hay que descargar nada al móvil. La app vive en el ordenador gratuito de Google.
- La app trae **vídeos de ejemplo** dentro (inglés y francés): podrás probarla sin grabar nada.
- Mientras uses la app, deja esta pestaña de Colab abierta (es el motor).
- Si sale un error en rojo: captura de pantalla y mándamela. Nada más.


In [ ]:
#@title 👉 TOCA EL ▶ DE LA IZQUIERDA Y ESPERA 10-15 MIN. NO HAY QUE HACER NADA MÁS { display-mode: "form" }
import os, sys, subprocess, warnings, traceback
warnings.filterwarnings("ignore")

try:
    print("🚀 Arrancando tu app de lectura de labios. Tarda 10-15 min la primera vez. NO cierres la página.\n")

    # ---------- 0) ¿Hay motor (GPU)? ----------
    try:
        gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        gpu = ""   # sin GPU ni siquiera existe nvidia-smi: seguimos, pero avisamos
    if "GPU" in gpu:
        print("✅ Motor encontrado:", gpu.splitlines()[0])
    else:
        print("⚠️  AVISO: no hay GPU activada. Funcionará, pero MUY lento.")
        print("   Para activarla: menú ⋮ → Entorno de ejecución → Cambiar tipo → T4 GPU → Guardar → vuelve a darle al ▶\n")

    # ---------- 1) Instalar piezas ----------
    print("📦 (1/6) Instalando piezas (2-4 min)...")
    os.system("pip install -q --upgrade mediapipe")
    os.system("pip install -q hydra-core av scikit-image sentencepiece six gradio")
    os.system("pip install -q -U gdown")   # versión reciente: sortea el aviso de Drive en ficheros grandes
    import mediapipe as mp
    print("   ✔ Piezas instaladas (mediapipe", mp.__version__ + ")")

    # ---------- 2) Descargar el cerebro (el repositorio del modelo) ----------
    print("📥 (2/6) Descargando el programa del modelo...")
    if not os.path.isdir("Visual_Speech_Recognition_for_Multiple_Languages"):
        os.system("git clone -q --depth 1 https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages")
    os.chdir("Visual_Speech_Recognition_for_Multiple_Languages")
    print("   ✔ Programa listo")

    # ---------- 3) Parches (arreglos para que el programa antiguo funcione en el Colab moderno) ----------
    print("🩹 (3/6) Aplicando arreglos de compatibilidad...")
    import numpy as np, torch, torchvision.io
    if not hasattr(torchvision.io, "read_video"):
        import av as _av
        def _read_video(filename, pts_unit="sec", **kw):
            c = _av.open(filename)
            fs = [f.to_ndarray(format="rgb24") for f in c.decode(video=0)]
            return torch.from_numpy(np.stack(fs)), None, {}
        torchvision.io.read_video = _read_video
    _tl = torch.load
    def _torch_load(*a, **k):
        k.setdefault("weights_only", False)
        return _tl(*a, **k)
    torch.load = _torch_load
    for _alias, _tipo in [("int", int), ("float", float), ("bool", bool), ("object", object)]:
        if not hasattr(np, _alias):
            setattr(np, _alias, _tipo)

    # (d) El mediapipe moderno ya no trae la API antigua "solutions" que usa el repo.
    #     Construimos un detector equivalente con la API nueva "tasks" y lo inyectamos.
    if not hasattr(mp, "solutions"):
        from mediapipe.tasks import python as mp_tasks
        from mediapipe.tasks.python import vision as mp_vision
        _MODELO_CARAS = "/content/blaze_face_short_range.tflite"
        if not os.path.exists(_MODELO_CARAS):
            import urllib.request as _ur
            for _u in ("https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
                       "https://raw.githubusercontent.com/tejex/faceAnonymizer/main/blaze_face_short_range.tflite"):
                try:
                    _ur.urlretrieve(_u, _MODELO_CARAS)
                    break
                except Exception:
                    continue
        assert os.path.exists(_MODELO_CARAS), "No pude bajar el modelito de caras. Captura y mándamela."

        class _DetectorModerno:
            """Misma salida que el detector antiguo del repo: por fotograma, None o
            un array 4x2 con [ojo dcho, ojo izq, nariz, boca] en píxeles."""
            def __init__(self):
                self._d = mp_vision.FaceDetector.create_from_options(
                    mp_vision.FaceDetectorOptions(
                        base_options=mp_tasks.BaseOptions(model_asset_path=_MODELO_CARAS),
                        min_detection_confidence=0.5))
            def __call__(self, filename):
                video = torchvision.io.read_video(filename, pts_unit="sec")[0].numpy()
                out = []
                for frame in video:
                    img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(frame))
                    res = self._d.detect(img)
                    if not res.detections:
                        out.append(None)
                        continue
                    ih, iw = frame.shape[:2]
                    mejor = max(res.detections, key=lambda d: d.bounding_box.width + d.bounding_box.height)
                    kp = mejor.keypoints
                    out.append(np.array([[int(kp[i].x * iw), int(kp[i].y * ih)] for i in range(4)]))
                return out

        import pipelines.detectors.mediapipe.detector as _det_mod
        _det_mod.LandmarksDetector = _DetectorModerno
        print("   ✔ Detector de caras adaptado a la API moderna de mediapipe")
    print("   ✔ Arreglos aplicados")

    # ---------- 4) Descargar los pesos del ESPAÑOL (≈1 GB, 3-6 min) ----------
    print("🧠 (4/6) Descargando el cerebro del ESPAÑOL (~1 GB, paciencia)...")
    import zipfile, configparser, urllib.request, gdown, time, shutil, re, requests

    IDIOMAS = {
        "es": {"nombre": "Español",   "config": "configs/CMUMOSEAS_V_ES_WER44.5.ini",
               "modelo": "https://bit.ly/34MjWBW", "lm": "https://bit.ly/3rppyJN"},
        "en": {"nombre": "Inglés",    "config": "configs/LRS3_V_WER19.1.ini",
               "modelo": "https://bit.ly/3Bp4gjV", "lm": "https://bit.ly/3qzWKit"},
        "fr": {"nombre": "Francés",   "config": "configs/CMUMOSEAS_V_FR_WER58.6.ini",
               "modelo": "https://bit.ly/3Ik6owb", "lm": "https://bit.ly/3LDChSn"},
        "pt": {"nombre": "Portugués", "config": "configs/CMUMOSEAS_V_PT_WER51.4.ini",
               "modelo": "https://bit.ly/3HjXCgo", "lm": "https://bit.ly/3gPvneF"},
        "zh": {"nombre": "Mandarín",  "config": "configs/CMLR_V_WER8.0.ini",
               "modelo": "https://bit.ly/3fR8RkU", "lm": "https://bit.ly/3fPxXAJ"},
        # Italiano: sin pesos públicos en este repo → no existe en el selector.
    }

    def _resolver(url):
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req) as r:
            return r.url

    def _drive_id(url):
        for patron in (r"/file/d/([\w-]+)", r"[?&]id=([\w-]+)", r"/d/([\w-]+)"):
            m = re.search(patron, url)
            if m:
                return m.group(1)
        return None

    def _descarga_manual_drive(file_id, dest):
        # Descargador robusto para ficheros grandes: sortea la página
        # "no puedo analizar en busca de virus" leyendo el token de confirmación.
        base = "https://drive.usercontent.google.com/download"
        s = requests.Session(); s.headers.update({"User-Agent": "Mozilla/5.0"})
        r = s.get(base, params={"id": file_id, "export": "download"}, stream=True)
        token = None
        for k, v in r.cookies.items():
            if k.startswith("download_warning"):
                token = v
        if token is None and "text/html" in r.headers.get("Content-Type", ""):
            cuerpo = r.text
            if "Too many users" in cuerpo or "quota" in cuerpo.lower():
                return "quota"
            m = re.search(r'name="confirm"\s+value="([^"]+)"', cuerpo) or re.search(r'confirm=([\w-]+)', cuerpo)
            if m:
                token = m.group(1)
        if token:
            r = s.get(base, params={"id": file_id, "export": "download", "confirm": token}, stream=True)
        with open(dest, "wb") as f:
            for trozo in r.iter_content(1 << 15):
                if trozo:
                    f.write(trozo)
        return "ok"

    def _bajar_zip(url, etiqueta):
        dest = f"/tmp/{etiqueta}.zip"
        ultimo = ""
        directo = _resolver(url)
        fid = _drive_id(directo)
        for intento in range(1, 5):
            try:
                if os.path.exists(dest):
                    os.remove(dest)
                gdown.download(url=directo, output=dest, quiet=True, fuzzy=True)
            except Exception as e:
                ultimo = str(e)
            # si gdown no dejó un ZIP válido, probamos el descargador manual
            if not (os.path.exists(dest) and zipfile.is_zipfile(dest)) and fid:
                try:
                    if _descarga_manual_drive(fid, dest) == "quota":
                        ultimo = "cuota de Google Drive"
                except Exception as e:
                    ultimo = str(e)
            if os.path.exists(dest) and zipfile.is_zipfile(dest):
                with zipfile.ZipFile(dest) as z:
                    z.extractall(".")
                return
            cab = b""
            if os.path.exists(dest):
                with open(dest, "rb") as fh:
                    cab = fh.read(600).lower()
            if b"<html" in cab or b"quota" in cab or b"too many users" in cab or ultimo == "cuota de Google Drive":
                print(f"      · Intento {intento}/4: Google Drive está limitando {etiqueta}, reintento...")
            else:
                print(f"      · Intento {intento}/4: {etiqueta} no llegó completo, reintento...")
            time.sleep(4 * intento)
        raise RuntimeError(
            f"No pude descargar {etiqueta} tras 4 intentos. Google Drive está limitando ese "
            f"archivo del modelo (demasiadas descargas). Espera 1-2 horas y vuelve a darle al ▶. "
            f"(detalle: {ultimo or 'el archivo no era un ZIP'})")

    def _colocar_en_ruta(ruta):
        # Si el ZIP trajo otra estructura de carpetas, localiza el archivo por su
        # nombre en todo el árbol y lo copia a donde el config lo espera.
        if os.path.exists(ruta):
            return True
        objetivo = os.path.basename(ruta)
        for raiz, _dirs, ficheros in os.walk("."):
            if objetivo in ficheros:
                os.makedirs(os.path.dirname(ruta) or ".", exist_ok=True)
                shutil.copy(os.path.join(raiz, objetivo), ruta)
                return True
        return False

    def preparar_idioma(cod):
        info = IDIOMAS[cod]
        cfg = configparser.ConfigParser(); cfg.read(info["config"])
        rutas = [cfg["model"]["model_path"], cfg["model"]["rnnlm"]]
        if not all(os.path.exists(r) for r in rutas):
            print(f"   ⬇ bajando {info['nombre']}...")
            _bajar_zip(info["modelo"], cod + "_modelo")
            _bajar_zip(info["lm"], cod + "_lm")
            for r in rutas:            # reubica si el ZIP traía otra estructura
                _colocar_en_ruta(r)
            faltan = [r for r in rutas if not os.path.exists(r)]
            if faltan:
                raise RuntimeError(
                    "El modelo se descargó pero faltan archivos tras descomprimir: "
                    + ", ".join(faltan)
                    + ". Suele ser la cuota de Google Drive: espera 1-2 horas y reintenta.")
        return info["config"]

    try:
        config_es = preparar_idioma("es")
        print("   ✔ Cerebro del español descargado")
    except Exception as e:
        config_es = None
        print("   ⚠ El español no se pudo bajar ahora:", str(e).splitlines()[0])
        print("     La app ABRIRÁ IGUAL. Elige el idioma en el menú y se baja al momento")
        print("     (si es la cuota de Google Drive, prueba en 1-2 h).")

    # ---------- 5) Encender el modelo + crear vídeos de ejemplo ----------
    print("⚙️ (5/6) Encendiendo el modelo y preparando vídeos de ejemplo...")
    from pipelines.pipeline import InferencePipeline
    from pipelines.detectors.mediapipe.detector import LandmarksDetector
    from pipelines.data.data_module import AVSRDataLoader
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    _pipes = {}
    if config_es:
        _pipes["es"] = InferencePipeline(config_es, detector="mediapipe", face_track=True, device=DEVICE)
    _detector = LandmarksDetector()
    _preview_loader = AVSRDataLoader(modality="video", detector="mediapipe", transform=False)

    import av
    def _gif_a_mp4(src, dst, maxf=150):
        entrada = av.open(src)
        fs = [f.to_ndarray(format="rgb24") for f in entrada.decode(video=0)][:maxf]
        salida = av.open(dst, "w")
        try: stream = salida.add_stream("h264", rate=25)
        except Exception: stream = salida.add_stream("mpeg4", rate=25)
        stream.width, stream.height = fs[0].shape[1], fs[0].shape[0]
        stream.pix_fmt = "yuv420p"
        for fr in fs:
            for pkt in stream.encode(av.VideoFrame.from_ndarray(fr, format="rgb24")):
                salida.mux(pkt)
        for pkt in stream.encode():
            salida.mux(pkt)
        salida.close()
    _gif_a_mp4("doc/vsr_1.gif", "/content/ejemplo_ingles.mp4")
    _gif_a_mp4("doc/vsr_2.gif", "/content/ejemplo_frances.mp4")
    print("   ✔ Modelo español encendido en", DEVICE, "y ejemplos creados")

    # ---------- 6) La app ----------
    print("🌐 (6/6) Levantando tu app con enlace público...\n")
    import gradio as gr
    from PIL import Image

    ACTIVOS = {"es": "Español", "en": "Inglés", "fr": "Francés", "pt": "Portugués", "zh": "Mandarín"}

    def _get_pipe(cod):
        if cod not in _pipes:
            cfg = preparar_idioma(cod)   # el primer uso de un idioma descarga su cerebro (tarda unos min)
            _pipes[cod] = InferencePipeline(cfg, detector="mediapipe", face_track=True, device=DEVICE)
        return _pipes[cod]

    def _preview(ruta):
        lm = _detector(ruta)
        roi = _preview_loader.load_data(ruta, lm)
        roi = np.asarray(roi.numpy() if hasattr(roi, "numpy") else roi)
        idx = np.linspace(0, len(roi) - 1, 5).astype(int)
        return Image.fromarray(np.hstack([roi[i] for i in idx]))

    def transcribir(video, idioma_nombre):
        if not video:
            return None, "Sube un vídeo o toca un ejemplo de abajo."
        cod = [k for k, v in ACTIVOS.items() if v == idioma_nombre][0]
        try:
            img = _preview(video)
        except Exception as e:
            return None, f"No encuentro una boca clara en el vídeo (¿de perfil? ¿poca luz?). Detalle: {e}"
        try:
            texto = _get_pipe(cod)(video)
        except Exception as e:
            return img, f"El idioma {idioma_nombre} no pudo cargarse ahora (posible cuota de Drive). Detalle: {e}"
        return img, texto

    with gr.Blocks(title="Lectura de labios") as demo:
        gr.Markdown("# 👄 Lectura de labios\n**1)** Sube un vídeo (cara de frente, buena luz) o toca un "
                    "ejemplo de abajo · **2)** elige idioma · **3)** toca Transcribir.\n\n"
                    "*Es una demo del estado del arte: se equivoca mucho, sobre todo en español (~la mitad "
                    "de las palabras). El primer uso de cada idioma tarda unos minutos (descarga su cerebro).*")
        with gr.Row():
            with gr.Column():
                v = gr.Video(label="Tu vídeo")
                idi = gr.Dropdown(choices=list(ACTIVOS.values()), value="Español", label="Idioma")
                btn = gr.Button("Transcribir 👄→📝", variant="primary")
            with gr.Column():
                img = gr.Image(label="Lo que el modelo ve (tu boca)")
                out = gr.Textbox(label="Transcripción", lines=4)
        gr.Examples(
            examples=[["/content/ejemplo_ingles.mp4", "Inglés"], ["/content/ejemplo_frances.mp4", "Francés"]],
            inputs=[v, idi], label="Ejemplos para probar sin grabar nada (toca uno y dale a Transcribir)")
        btn.click(transcribir, [v, idi], [img, out])

    print("=" * 60)
    print("🎉 ¡LISTO! Tu app está encendida.")
    print("👇 Toca el enlace que termina en .gradio.live (aquí abajo)")
    print("=" * 60)
    demo.launch(share=True, quiet=True)

except Exception:
    print("\n" + "❌" * 20)
    print("Algo ha fallado. HAZ UNA CAPTURA DE PANTALLA de lo de abajo y mándasela a Claude:")
    print("❌" * 20)
    traceback.print_exc()

---
### ¿Problemas?
- **No sale el enlace y hay letras rojas** → captura de pantalla y mándamela.
- **Dice "cuota de Google Drive"** → los dueños del modelo lo comparten por Drive y a veces se
  llena el cupo diario; espera unas horas y vuelve a darle al ▶.
- **La página se quedó dormida** (Colab desconecta si la dejas mucho tiempo) → vuelve a darle al ▶.
- **El enlace gradio.live caduca a las 72 h** → dale al ▶ otra vez y sale uno nuevo.
